In [1]:
import gdown

file_id = '17VNJPetsxVsQDvj5q8K-8bunOpq1akUF'
output_filename = 'processos_simulacao.csv'
gdown.download(id=file_id, output=output_filename, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=17VNJPetsxVsQDvj5q8K-8bunOpq1akUF
To: /content/processos_simulacao.csv
100%|██████████| 1.26k/1.26k [00:00<00:00, 844kB/s]


'processos_simulacao.csv'

In [2]:
import csv
from collections import deque

class Processo:
    def __init__(self, id_processo, tempo_chegada, tempo_execucao, prioridade):
        self.id = id_processo
        self.tempo_chegada = int(tempo_chegada)
        self.tempo_execucao_total = int(tempo_execucao)
        self.tempo_restante = int(tempo_execucao)
        self.prioridade = int(prioridade)

        # Métricas de saída
        self.primeiro_inicio = -1
        self.tempo_termino = -1
        self.tempo_espera = 0
        self.tempo_retorno = 0

def carregar_processos(caminho_csv):
    processos = []
    with open(caminho_csv, mode='r', encoding='utf-8') as arquivo:
        leitor = csv.DictReader(arquivo)
        for linha in leitor:
            proc = Processo(
                linha['id_processo'].strip(),
                linha['tempo_chegada'].strip(),
                linha['tempo_execucao'].strip(),
                linha['prioridade'].strip()
            )
            processos.append(proc)

    # Ordena por tempo de chegada
    return sorted(processos, key=lambda x: x.tempo_chegada)

def simular_escalonamento(processos_lista, quantum, troca_contexto=0):
    tempo_global = 0
    processos_restantes = processos_lista.copy()

    # 3 Filas de Prioridade (Prioridade 1, 2 e 3)
    filas = {1: deque(), 2: deque(), 3: deque()}

    processos_finalizados = []
    ultimo_processo_executado = None

    # Loop continua enquanto houver processos para chegar ou filas para processar
    while processos_restantes or any(filas.values()):

        # Atualiza as filas com processos que chegaram
        chegadas = [p for p in processos_restantes if p.tempo_chegada <= tempo_global]
        for p in chegadas:
            filas[p.prioridade].append(p)
            processos_restantes.remove(p)

        if not any(filas.values()) and processos_restantes:
            tempo_global = processos_restantes[0].tempo_chegada
            continue

        # Escolhe a fila de maior prioridade que não esteja vazia (1 > 2 > 3)
        fila_atual_id = 1
        if filas[1]:
            fila_atual_id = 1
        elif filas[2]:
            fila_atual_id = 2
        elif filas[3]:
            fila_atual_id = 3

        processo_atual = filas[fila_atual_id].popleft()

        if ultimo_processo_executado and ultimo_processo_executado != processo_atual.id and troca_contexto > 0:
            tempo_global += troca_contexto

            # Durante a troca de contexto, verifica se novos processos chegaram
            chegadas_contexto = [p for p in processos_restantes if p.tempo_chegada <= tempo_global]
            for p in chegadas_contexto:
                filas[p.prioridade].append(p)
                processos_restantes.remove(p)

        # Registra o tempo do primeiro início na CPU
        if processo_atual.primeiro_inicio == -1:
            processo_atual.primeiro_inicio = tempo_global

        # Define quanto tempo o processo vai rodar neste quantum
        tempo_rodado = min(processo_atual.tempo_restante, quantum)

        # Executa (aumenta o tempo global e reduz o tempo restante do processo)
        tempo_global += tempo_rodado
        processo_atual.tempo_restante -= tempo_rodado
        ultimo_processo_executado = processo_atual.id

        # Verifica se novos processos chegaram enquanto este estava executando
        chegadas_durante = [p for p in processos_restantes if p.tempo_chegada <= tempo_global]
        for p in chegadas_durante:
            filas[p.prioridade].append(p)
            processos_restantes.remove(p)

        # Verifica se o processo terminou ou volta para a fila
        if processo_atual.tempo_restante == 0:
            processo_atual.tempo_termino = tempo_global
            processo_atual.tempo_retorno = processo_atual.tempo_termino - processo_atual.tempo_chegada
            processo_atual.tempo_espera = processo_atual.tempo_retorno - processo_atual.tempo_execucao_total
            processos_finalizados.append(processo_atual)
        else:
            # Se não terminou, volta para o final da fila de prioridade
            filas[fila_atual_id].append(processo_atual)

    return processos_finalizados

def exibir_resultados(processos_resultados):
    # Ordena pelo ID numérico para exibição organizada na tabela
    processos_resultados.sort(key=lambda x: int(x.id.replace('P', '')))

    print("\n" + "="*85)
    print(f"{'ID':<7}{'Chegada':<10}{'1º Início':<12}{'Término':<10}{'Espera (TE)':<15}{'Retorno (TR)':<15}")
    print("="*85)

    soma_te = 0
    soma_tr = 0

    for p in processos_resultados:
        print(f"{p.id:<7}{p.tempo_chegada:<10}{p.primeiro_inicio:<12}{p.tempo_termino:<10}{p.tempo_espera:<15}{p.tempo_retorno:<15}")
        soma_te += p.tempo_espera
        soma_tr += p.tempo_retorno

    total_p = len(processos_resultados)
    tme = soma_te / total_p
    tmr = soma_tr / total_p

    print("="*85)
    print(f"Tempo Médio de Espera (TME) : {tme:.2f}")
    print(f"Tempo Médio de Retorno (TMR): {tmr:.2f}")
    print("="*85)

In [3]:
if __name__ == "__main__":
    caminho_arquivo = "processos_simulacao.csv"

    print("--- SIMULADOR DE ESCALONAMENTO ROUND ROBIN COM PRIORIDADE ---")
    try:
        quantum_input = int(input("Informe o valor do Quantum: "))
        troca_input = int(input("Informe o Tempo de Troca de Contexto: "))

        lista_de_processos = carregar_processos(caminho_arquivo)

        resultados = simular_escalonamento(lista_de_processos, quantum_input, troca_input)

        exibir_resultados(resultados)

    except FileNotFoundError:
        print(f"Erro: Certifique-se de que o arquivo '{caminho_arquivo}' está na mesma pasta deste script.")
    except ValueError:
        print("Erro: Por favor, insira valores inteiros válidos para o simulador.")

--- SIMULADOR DE ESCALONAMENTO ROUND ROBIN COM PRIORIDADE ---
Informe o valor do Quantum: 5
Informe o Tempo de Troca de Contexto: 6

ID     Chegada   1º Início   Término   Espera (TE)    Retorno (TR)   
P1     0         0           2414      2397           2414           
P2     1         11          1384      1367           1383           
P3     17        27          395       359            378            
P4     18        1048        1377      1353           1359           
P5     20        38          552       513            532            
P6     23        49          130       98             107            
P7     27        1070        1393      1358           1366           
P8     37        1081        1763      1706           1726           
P9     37        71          184       138            147            
P10    38        1092        1774      1716           1736           
P11    39        1103        1784      1726           1745           
P12    41        82        